# v0.5.0 Reviewer-Grade E2E: HSC CMP vs CD14+ MonocyteFull PEACH pipeline with statistical controls, interleaved visualizations,HALLMARK pathway scoring, and flow-aligned gene/geneset analysis.**Pipeline:**1. Data loading with gene symbol mapping2. Archetype fitting (K=4) with training diagnostics3. Simplex regression with bootstrap CIs + permutation tests4. Feature pattern classification5. GMM simplex decomposition6. Archetype comparison (MMD, similarity, Wald contrasts)7. HALLMARK pathway scoring & regression8. Flow matching with gene & geneset alignment9. Cross-validation & summary**Dataset:** Human Hematopoietic Stem Cells (263K cells) — CMP vs CD14+ Monocyte comparison

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["MKL_NUM_THREADS"] = "1"

import time
import warnings
import logging
import numpy as np
import scipy.sparse as sp
import pandas as pd
import anndata as ad
import gc
from collections import Counter

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=PendingDeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
logging.disable(logging.WARNING)

import torch
torch.set_num_threads(1)

import peach as pc

# --- Config ---
HSC_PATH = "/Users/honkala/Desktop/cross_recons/data/HSC.h5ad"
K = 4
N_PCS = 20
MONO_SUBSAMPLE = 9000
TRAIN_EPOCHS = 150
FLOW_EPOCHS = 300
SEED = 42
N_BOOTSTRAP = 50        # increase to 200-1000 for publication
N_PERMUTATIONS = 50      # increase to 200-1000 for publication
ALPHA = 0.05

t0 = time.time()

## 1. Data Loading & Gene Symbol MappingLoad HSC dataset, swap ENSG IDs to gene symbols, subset CMP and Monocyte populations.

In [ ]:
adata_full = ad.read_h5ad(HSC_PATH)
print(f"Loaded: {adata_full.shape[0]:,} cells x {adata_full.shape[1]:,} genes")
n_genes_original = adata_full.n_vars

# Swap ENSG -> gene symbols (handle NaN, duplicates)
symbols_raw = list(adata_full.var["gene_symbols"].values)
symbols = []
for i, s in enumerate(symbols_raw):
    if pd.isna(s) or str(s).strip() == "":
        symbols.append(f"UNNAMED_{i}")
    else:
        symbols.append(str(s).strip())

counts = Counter(symbols)
seen = Counter()
unique_symbols = []
for s in symbols:
    if counts[s] > 1:
        seen[s] += 1
        unique_symbols.append(f"{s}_{seen[s]}")
    else:
        unique_symbols.append(s)
var_names = unique_symbols

print(f"Gene symbols: {len(var_names)} total, {len(set(symbols))} unique")
print(f"Sample: {list(adata_full.var_names[:3])} -> {var_names[:3]}")
assert len(var_names) == n_genes_original
assert len(var_names) == len(set(var_names))

# Extract PCA loadings
if "PCs" in adata_full.varm:
    pca_loadings = adata_full.varm["PCs"][:, :N_PCS].copy()
    print(f"PCA loadings from dataset: {pca_loadings.shape}")
else:
    pca_loadings = None
    print("PCA loadings not in dataset -- will compute from CMP subset")

# Subset populations
rng = np.random.default_rng(SEED)
cell_data = {}
for name, ct, subsample in [
    ("CMP", "common myeloid progenitor", None),
    ("Mono", "CD14-positive monocyte", MONO_SUBSAMPLE),
]:
    mask = adata_full.obs["cell_type"] == ct
    idx = np.where(mask)[0]
    n_total = len(idx)
    if subsample and len(idx) > subsample:
        idx = rng.choice(idx, size=subsample, replace=False)
        idx.sort()
    pca = adata_full.obsm["X_pca"][idx, :N_PCS].copy()
    obs = adata_full.obs.iloc[idx].copy().reset_index(drop=True)
    cell_data[name] = {"pca": pca, "obs": obs, "idx": idx}
    sub = f" (subsampled from {n_total:,})" if subsample else ""
    print(f"{name}: {len(idx):,} cells{sub}, PCA: {pca.shape}")

# Compute PCA loadings if not available
if pca_loadings is None:
    from sklearn.decomposition import PCA
    X_for_pca = adata_full.X[cell_data["CMP"]["idx"]]
    if hasattr(X_for_pca, "toarray"):
        X_for_pca = X_for_pca.toarray()
    pca_model = PCA(n_components=N_PCS, random_state=SEED)
    pca_model.fit(X_for_pca)
    pca_loadings = pca_model.components_.T
    del X_for_pca
    print(f"Computed PCA loadings: {pca_loadings.shape}")

del adata_full
gc.collect()
print("Data loading complete.")

In [ ]:
# --- Data structure inspection ---
print("=== cell_data layout ===")
for name in ["CMP", "Mono"]:
    d = cell_data[name]
    print(f"\n  {name}:")
    print(f"    pca:  {type(d['pca']).__name__}  {d['pca'].shape}  dtype={d['pca'].dtype}")
    print(f"    obs:  DataFrame  {d['obs'].shape}  columns={list(d['obs'].columns[:6])}")
    print(f"    idx:  {type(d['idx']).__name__}  len={len(d['idx'])}")

print(f"\n  pca_loadings: {type(pca_loadings).__name__}  {pca_loadings.shape}  dtype={pca_loadings.dtype}")
print(f"  var_names: list  len={len(var_names)}  sample={var_names[:3]}")

## 2. Archetype Fitting (K=4)Train Deep Archetypal Analysis on both cell types. Extract weights, coordinates, and assign archetypes.**Positive controls:** R2 > 0.5, weights sum to 1, all archetypes have dominant cells.

In [ ]:
models = {}
for name in ["CMP", "Mono"]:
    print(f"\n--- Training K={K} on {name} ---")
    t1 = time.time()

    adata_train = ad.AnnData(
        X=sp.csr_matrix((len(cell_data[name]["pca"]), 1)),
        obs=cell_data[name]["obs"].copy(),
    )
    adata_train.obsm["X_pca"] = cell_data[name]["pca"]

    result = pc.tl.train_archetypal(
        adata_train, n_archetypes=K,
        n_epochs=TRAIN_EPOCHS, kld_weight=0.1, archetypal_weight=0.9,
        seed=SEED,
    )
    elapsed = time.time() - t1
    r2 = result.get("final_archetype_r2", None)
    print(f"R2={r2:.3f}, {elapsed:.1f}s")

    pc.tl.extract_archetype_weights(adata_train)
    pc.tl.archetypal_coordinates(adata_train, verbose=False)

    weights = adata_train.obsm["cell_archetype_weights"]
    cell_data[name]["weights"] = weights
    cell_data[name]["uns"] = dict(adata_train.uns)
    cell_data[name]["arch_dist"] = adata_train.obsm["archetype_distances"]
    models[name] = result

    # Diagnostics
    print(f"  Weight sum: {weights.sum(axis=1).mean():.6f}")
    print(f"  Weight range: [{weights.min():.4f}, {weights.max():.4f}]")
    for k in range(K):
        n_dom = int(np.sum(weights.argmax(axis=1) == k))
        print(f"  Archetype {k}: {n_dom} dominant cells ({100*n_dom/len(weights):.1f}%)")

    # Assertions
    assert r2 is not None and r2 > 0.0, f"Degenerate model: R2={r2}"
    assert np.allclose(weights.sum(axis=1), 1.0, atol=1e-4)
    for k_idx in range(K):
        assert int(np.sum(weights.argmax(axis=1) == k_idx)) > 0, f"Archetype {k_idx} has zero dominant cells"

In [ ]:
# --- Training result structure inspection ---
for name in ["CMP", "Mono"]:
    result = models[name]
    print(f"\n=== {name} TrainingResults ===")
    print(f"  type: {type(result).__name__}")
    print(f"  keys: {list(result.keys())}")
    for k, v in result.items():
        if isinstance(v, np.ndarray):
            print(f"    {k}: ndarray {v.shape} dtype={v.dtype}")
        elif isinstance(v, (list, tuple)):
            print(f"    {k}: {type(v).__name__} len={len(v)}")
        else:
            print(f"    {k}: {type(v).__name__} = {v}")

    w = cell_data[name]["weights"]
    print(f"  weights: {w.shape} dtype={w.dtype}")
    print(f"    row sums: mean={w.sum(1).mean():.6f}, std={w.sum(1).std():.2e}")
    print(f"    min/max:  [{w.min():.4f}, {w.max():.4f}]")
    print(f"    sparsity: {(w < 0.01).sum() / w.size:.1%} of entries < 0.01")

In [ ]:
# Reload sparse X with gene symbol var_names for downstream analysis
print("Rebuilding adata with gene symbols and sparse X...")
adata_disk = ad.read_h5ad(HSC_PATH, backed="r")
for name in ["CMP", "Mono"]:
    idx = cell_data[name]["idx"]
    X_sparse = sp.csr_matrix(adata_disk.X[idx])
    var_df = pd.DataFrame(index=var_names)

    adata_obj = ad.AnnData(X=X_sparse, obs=cell_data[name]["obs"].copy(), var=var_df)
    adata_obj.obsm["X_pca"] = cell_data[name]["pca"]
    adata_obj.obsm["cell_archetype_weights"] = cell_data[name]["weights"]
    adata_obj.obsm["archetype_distances"] = cell_data[name]["arch_dist"]
    for k, v in cell_data[name]["uns"].items():
        adata_obj.uns[k] = v
    if pca_loadings is not None:
        adata_obj.varm["PCs"] = pca_loadings
    pc.tl.assign_archetypes(adata_obj)

    assert adata_obj.shape == (len(idx), len(var_names))
    assert "archetypes" in adata_obj.obs.columns
    cell_data[name]["adata"] = adata_obj

del adata_disk
gc.collect()

adata_cmp = cell_data["CMP"]["adata"]
adata_mono = cell_data["Mono"]["adata"]
print(f"CMP: {adata_cmp.shape}, Mono: {adata_mono.shape}")
print(f"var_names sample: {list(adata_cmp.var_names[:3])}")

In [ ]:
# --- Adata structure inspection ---
for name, ad_obj in [("CMP", adata_cmp), ("Mono", adata_mono)]:
    print(f"\n=== {name} adata ===")
    print(f"  X:       {type(ad_obj.X).__name__}  {ad_obj.X.shape}  dtype={ad_obj.X.dtype}")
    print(f"  obs:     {ad_obj.obs.shape}  columns={list(ad_obj.obs.columns)}")
    print(f"  var:     {ad_obj.var.shape}  index[:3]={list(ad_obj.var_names[:3])}")
    print(f"  obsm:    {list(ad_obj.obsm.keys())}")
    for k in ad_obj.obsm:
        print(f"           {k}: {ad_obj.obsm[k].shape}")
    print(f"  varm:    {list(ad_obj.varm.keys())}")
    print(f"  uns:     {list(ad_obj.uns.keys())}")

## 3. Simplex Regression

Scheffe polynomial regression of gene expression on archetype weights.

- **Degree 1**: Linear dependence on archetype membership
- **Degree 2**: Interaction terms (synergistic archetype effects)
- **Comprehensive degree analysis**: Incremental F-tests comparing degree d vs d-1
- **Bootstrap CIs**: Confidence intervals on vertex coefficients
- **Permutation test**: Non-parametric significance (shuffles weight-cell correspondence)
- **Robust SE**: HC3 heteroscedasticity-consistent standard errors
- **Vertex Wald tests**: Per-archetype t-tests on individual coefficients (FDR-corrected)

**Statistical controls:**
- R2 bounds: degree-1 in [-0.1, 1], degree-2 in [-0.1, 1], finite
- Nested model invariant: degree-2 R2 >= degree-1 R2
- F-test / permutation concordance
- Degree separation: incremental F-test quantifies added value of interaction terms

In [ ]:
gene_regs = {}
for name, ad_obj in [("CMP", adata_cmp), ("Mono", adata_mono)]:
    print(f"\n--- Simplex regression on {name} (bootstrap={N_BOOTSTRAP}, perm={N_PERMUTATIONS}) ---")
    t1 = time.time()
    reg = pc.tl.feature_simplex_regression(
        ad_obj, max_degree=2,
        n_bootstrap=N_BOOTSTRAP,
        permutation_test=True,
        n_permutations=N_PERMUTATIONS,
        robust_se=True,
        comprehensive_degree=True,
    )
    elapsed = time.time() - t1
    print(f"Completed in {elapsed:.1f}s")
    gene_regs[name] = reg

    r2_d1 = np.asarray(reg["r_squared_degree1"])
    r2_d2 = np.asarray(reg["r_squared_degree2"])
    f_pval = np.asarray(reg["f_pvalue"])
    f_pval_fdr = np.asarray(reg["f_pvalue_fdr"])
    perm_pval_fdr = reg.get("permutation_pvalue_fdr")
    if perm_pval_fdr is not None:
        perm_pval_fdr = np.asarray(perm_pval_fdr)

    n_sig_f = int(np.sum(f_pval_fdr < ALPHA))
    n_sig_perm = int(np.sum(perm_pval_fdr < ALPHA)) if perm_pval_fdr is not None else "N/A"

    print(f"Features: {len(r2_d1)}")
    print(f"Degree-1 R2: median={np.median(r2_d1):.4f}, max={np.max(r2_d1):.4f}")
    print(f"Degree-2 R2: median={np.median(r2_d2):.4f}, max={np.max(r2_d2):.4f}")
    print(f"F-test significant (FDR<{ALPHA}): {n_sig_f}/{len(r2_d1)} ({100*n_sig_f/len(r2_d1):.1f}%)")
    print(f"Permutation significant (FDR<{ALPHA}): {n_sig_perm}")

    print(f"\nTOP 10 GENES (by R2):")
    top_idx = np.argsort(r2_d1)[-10:][::-1]
    for i in top_idx:
        print(f"  {reg['feature_names'][i]}: R2={r2_d1[i]:.4f}, F_p={f_pval[i]:.2e}")

    # Assertions
    assert len(r2_d1) == ad_obj.n_vars
    assert np.all(r2_d1 >= -0.1) and np.all(r2_d1 <= 1.0 + 1e-6)
    assert np.all(np.isfinite(r2_d2))
    assert np.all(r2_d2 >= -0.1) and np.all(r2_d2 <= 1.0 + 1e-6)
    r2_diff = r2_d2 - r2_d1
    assert int(np.sum(r2_diff < -1e-6)) == 0, "Nested model invariant violated"
    assert n_sig_f > 0
    if perm_pval_fdr is not None:
        concordance = min(n_sig_f, n_sig_perm) / max(n_sig_f, n_sig_perm)
        print(f"F-test/permutation concordance: {concordance:.2f}")

In [ ]:
# --- Regression result structure inspection ---
for name in ["CMP", "Mono"]:
    reg = gene_regs[name]
    print(f"\n=== {name} SimplexRegressionResult ===")
    print(f"  type: {type(reg).__name__}")
    print(f"  keys: {sorted(reg.keys())}")
    for k, v in sorted(reg.items()):
        if isinstance(v, dict):
            print(f"    {k}: dict ({len(v)} keys)")
        elif isinstance(v, (list, np.ndarray)):
            v = np.asarray(v)
            if v.dtype.kind in ("f", "i", "u"):  # numeric
                print(f"    {k}: {v.shape} dtype={v.dtype}  range=[{v.min():.4f}, {v.max():.4f}]")
            else:
                print(f"    {k}: {v.shape} dtype={v.dtype}")
        elif isinstance(v, (int, float)):
            print(f"    {k}: {v}")
    ad_obj = adata_cmp if name == "CMP" else adata_mono
    print(f"\n  adata.uns keys after regression: {sorted(ad_obj.uns.keys())}")
    for uk in sorted(ad_obj.uns.keys()):
        if "simplex" in uk:
            print(f"    {uk}: {type(ad_obj.uns[uk]).__name__} ({len(ad_obj.uns[uk])} keys)")

### 3a. Vertex Wald Tests & Degree Separation

**Vertex Wald tests** assess whether each archetype's coefficient is individually significant
(t-test on each beta_k, FDR-corrected across all genes x archetypes).

**Degree separation** quantifies how much the interaction terms (degree 2) improve fit over
the linear model (degree 1) via incremental F-tests.

In [ ]:
# 3a. Vertex Wald tests (per-archetype coefficient significance)
for name in ["CMP", "Mono"]:
    reg = gene_regs[name]
    vertex_pvals_fdr = np.asarray(reg["vertex_pvalues_fdr"])
    vertex_se = np.asarray(reg["vertex_se"])
    coefs = np.asarray(reg["vertex_coefficients"])

    print(f"\n{'='*60}")
    print(f"  Vertex Wald Tests -- {name}")
    print(f"{'='*60}")
    for k in range(K):
        n_sig_k = int(np.sum(vertex_pvals_fdr[:, k] < ALPHA))
        print(f"  Archetype {k}: {n_sig_k} genes significant (FDR<{ALPHA})")
        # Top 5 by |beta/SE| (Wald statistic)
        z_stat = np.abs(coefs[:, k]) / np.maximum(vertex_se[:, k], 1e-10)
        top5_k = np.argsort(z_stat)[-5:][::-1]
        for i in top5_k:
            print(f"    {reg['feature_names'][i]:20s}  beta={coefs[i, k]:.3f}  SE={vertex_se[i, k]:.3f}  z={z_stat[i]:.1f}  p_fdr={vertex_pvals_fdr[i, k]:.2e}")

    assert vertex_pvals_fdr.shape == (len(reg["feature_names"]), K)
    assert np.all(vertex_se >= 0)

# 3b. Degree separation (incremental F-test: degree 2 vs degree 1)
for name in ["CMP", "Mono"]:
    reg = gene_regs[name]
    deg_comp = reg.get("degree_comparison")

    print(f"\n{'='*60}")
    print(f"  Degree Separation -- {name}")
    print(f"{'='*60}")
    if deg_comp is None:
        print("  (comprehensive_degree=False, skipping)")
        continue
    for deg_key, deg_info in sorted(deg_comp.items()):
        delta_r2 = np.asarray(deg_info["delta_r2"])
        inc_p_fdr = np.asarray(deg_info["incremental_p_fdr"])
        n_sig_inc = int(deg_info["significant_features"])
        n_features = len(delta_r2)
        print(f"\n  {deg_key}: {deg_info['n_params']} params, {deg_info['df_extra']} extra df")
        print(f"    Genes with significant interaction improvement: {n_sig_inc}/{n_features} ({100*n_sig_inc/n_features:.1f}%)")
        print(f"    Delta R2: median={np.median(delta_r2):.6f}, max={np.max(delta_r2):.4f}")
        # Top 5 genes gaining most from interactions
        top5_deg = np.argsort(delta_r2)[-5:][::-1]
        print(f"    TOP 5 genes (by delta R2):")
        for i in top5_deg:
            print(f"      {reg['feature_names'][i]:20s}  dR2={delta_r2[i]:.4f}  p_fdr={inc_p_fdr[i]:.2e}")

    assert deg_comp is not None, "comprehensive_degree=True but no degree_comparison in result"

In [ ]:
# Regression visualizations -- CMP
_ = pc.pl.coefficient_heatmap(adata_cmp, top_n=30, show=True)
_ = pc.pl.r2_barplot(adata_cmp, top_n=30, show=True)
_ = pc.pl.regression_volcano(adata_cmp, show=True)

In [ ]:
# Regression visualizations -- Mono
_ = pc.pl.coefficient_heatmap(adata_mono, top_n=30, show=True)
_ = pc.pl.r2_barplot(adata_mono, top_n=30, show=True)
_ = pc.pl.regression_volcano(adata_mono, show=True)

## 4. Pattern Classification

Classify features into simplex patterns from regression coefficients.

**Classification criteria (checked in priority order):**

| Pattern | Rule | Interpretation |
|---------|------|----------------|
| **flat** | R² < 0.05 OR CV(beta) < 0.15 | No archetype dependence |
| **archetype-exclusive** | max(\|beta\|) / second-max(\|beta\|) >= 2.0 | Marker gene for one archetype |
| **gradient** | 2+ betas in top tier, large gap (>30% of range) from rest | Shared program across archetype subset |
| **monotonic** | Fallback for structured, diffuse patterns | Graded expression across archetypes |

**Negative control:** >80% of genes should be flat (not archetype-dependent).

In [ ]:
pattern_results = {}
for name, ad_obj in [("CMP", adata_cmp), ("Mono", adata_mono)]:
    reg = gene_regs[name]
    patterns = pc.tl.classify_feature_patterns(ad_obj)
    pattern_results[name] = patterns
    counts_p = patterns["pattern_counts"]
    n_total = patterns["n_features"]

    print(f"\n{'='*60}")
    print(f"  Pattern Classification -- {name}")
    print(f"{'='*60}")
    for ptype, pcount in sorted(counts_p.items(), key=lambda x: -x[1]):
        bar = "\u2588" * int(50 * pcount / n_total)
        print(f"  {ptype:25s} {pcount:5d} ({100*pcount/n_total:5.1f}%) {bar}")

    n_flat = counts_p.get("flat", 0)
    print(f"\n  Flat control: {100*n_flat/n_total:.1f}% (expect >80%)")

    # Report top features per non-flat pattern
    classifications = patterns["classifications"]
    feat_names = patterns["feature_names"]
    r2 = np.asarray(reg["r_squared_degree1"])
    coefs = np.asarray(reg["vertex_coefficients"])

    for ptype in sorted(counts_p.keys()):
        if ptype == "flat":
            continue
        indices = [i for i, c in enumerate(classifications) if c["pattern"] == ptype]
        if not indices:
            continue
        indices_sorted = sorted(indices, key=lambda i: r2[i], reverse=True)
        top5 = indices_sorted[:5]
        print(f"\n  --- Top {ptype} features (by R2) ---")
        for i in top5:
            betas = coefs[i]
            beta_str = ", ".join(f"{b:.3f}" for b in betas)
            details = classifications[i].get("details", {})
            detail_str = " ".join(f"{k}={v}" for k, v in details.items())
            print(f"    {feat_names[i]:20s}  R2={r2[i]:.4f}  beta=[{beta_str}]  {detail_str}")

    assert n_total == ad_obj.n_vars
    assert sum(counts_p.values()) == n_total
    assert n_flat > 0

## 5. GMM Simplex DecompositionFit Gaussian Mixture Model in ILR-transformed weight space to identify cell subpopulations within the simplex.**ILR (Isometric Log-Ratio) transform**: Maps K-dimensional simplex compositions to (K-1)-dimensionalunconstrained Euclidean space via `ILR(w) = log(w + e) @ V`, where V is the Helmert sub-matrixand e=1e-3 smooths boundary cells.**Sanity checks:**- ILR roundtrip fidelity (weights -> ILR -> inverse -> weights)- ILR value distribution (no extreme outliers from boundary cells)- GMM component stability across random initializations

In [ ]:
from peach._core.utils.ilr_transform import ilr_transform, inverse_ilr

gmm_results = {}
for name, ad_obj in [("CMP", adata_cmp), ("Mono", adata_mono)]:
    weights = ad_obj.obsm["cell_archetype_weights"]

    # --- ILR sanity checks ---
    print(f"\n{'='*60}")
    print(f"  ILR Sanity -- {name}")
    print(f"{'='*60}")

    ilr_coords = ilr_transform(weights)
    roundtrip = inverse_ilr(ilr_coords)
    roundtrip_error = np.abs(roundtrip - weights).max()
    print(f"  ILR shape:     {ilr_coords.shape}  (expect [{weights.shape[0]}, {K-1}])")
    print(f"  ILR range:     [{ilr_coords.min():.3f}, {ilr_coords.max():.3f}]")
    print(f"  ILR mean:      {ilr_coords.mean(0)}")
    print(f"  ILR std:       {ilr_coords.std(0)}")
    print(f"  Roundtrip err: {roundtrip_error:.2e}  (expect < 1e-2 with epsilon smoothing)")

    n_extreme = int(np.sum(np.abs(ilr_coords) > 5))
    print(f"  |ILR| > 5:     {n_extreme} entries ({100*n_extreme/ilr_coords.size:.2f}%)")

    assert ilr_coords.shape == (weights.shape[0], K - 1)
    # Epsilon-smoothed ILR has bounded roundtrip error (~O(epsilon))
    assert roundtrip_error < 0.01, f"ILR roundtrip failed: err={roundtrip_error}"
    assert np.all(np.isfinite(ilr_coords)), "ILR contains non-finite values"

    # --- GMM fitting ---
    print(f"\n  --- GMM ({name}) ---")
    t1 = time.time()
    gmm = pc.tl.feature_simplex_decomposition(ad_obj, characterize_features=True)
    elapsed = time.time() - t1
    gmm_results[name] = gmm

    print(f"  Completed in {elapsed:.1f}s")
    print(f"  Optimal components (BIC): {gmm['n_components_optimal']}")
    print(f"  Stable components: {gmm['n_components_stable']}")
    print(f"  Archetype map: {gmm['component_archetype_map']}")

    if "stability_scores" in gmm:
        stab = np.asarray(gmm["stability_scores"])
        print(f"  Stability scores: {np.array2string(stab, precision=3)}")
        print(f"    Mean: {stab.mean():.3f}  Min: {stab.min():.3f}  Max: {stab.max():.3f}")

    assert gmm["n_components_optimal"] >= K
    assert gmm["n_components_stable"] >= 1

In [ ]:
# --- GMM result structure inspection ---
for name in ["CMP", "Mono"]:
    gmm = gmm_results[name]
    print(f"\n=== {name} GMMResult ===")
    print(f"  type: {type(gmm).__name__}")
    print(f"  keys: {sorted(gmm.keys())}")
    for k, v in sorted(gmm.items()):
        if isinstance(v, np.ndarray):
            print(f"    {k}: ndarray {v.shape}")
        elif isinstance(v, list):
            if len(v) > 0 and isinstance(v[0], (int, float, np.integer, np.floating)):
                print(f"    {k}: list len={len(v)}")
            else:
                print(f"    {k}: list len={len(v)}")
        elif isinstance(v, dict):
            print(f"    {k}: dict keys={list(v.keys())[:5]}")
        else:
            print(f"    {k}: {type(v).__name__} = {v}")

In [ ]:
# GMM visualizations (CMP)
_ = pc.pl.component_scatter(adata_cmp, show=True)
_ = pc.pl.gmm_bic_curve(adata_cmp, show=True)
_ = pc.pl.component_heatmap(adata_cmp, top_n=30, show=True)
_ = pc.pl.component_stability(adata_cmp, show=True)

### 5a. GMM Component-Archetype Distance

Map GMM components to their location in archetype weight space. Each component's centroid
(mean archetype weights of assigned cells) is compared to the K archetype vertices
(unit vectors on the simplex). This shows which archetypes each component is closest to
and how far into the interior of the simplex they sit.

In [ ]:
import plotly.graph_objects as go

# Archetype vertices on K-simplex (unit vectors)
archetype_vertices = np.eye(K)  # e_k = [0,...,1,...,0]
# Simplex center (equal weights)
simplex_center = np.ones(K) / K

for name in ["CMP", "Mono"]:
    gmm = gmm_results[name]
    weight_means = gmm.get("component_weight_means")
    if weight_means is None:
        print(f"  {name}: component_weight_means not available")
        continue

    n_comp = weight_means.shape[0]
    arch_map = gmm["component_archetype_map"]

    # Euclidean distance from each component centroid to each archetype vertex
    dist_matrix = np.zeros((n_comp, K))
    for c in range(n_comp):
        for k in range(K):
            dist_matrix[c, k] = np.linalg.norm(weight_means[c] - archetype_vertices[k])
    # Distance to simplex center
    dist_to_center = np.array([np.linalg.norm(weight_means[c] - simplex_center) for c in range(n_comp)])

    print(f"\n{'='*60}")
    print(f"  GMM Component-Archetype Distance -- {name}")
    print(f"{'='*60}")
    print(f"  Components: {n_comp}")
    for c in range(n_comp):
        w_str = ", ".join(f"{w:.3f}" for w in weight_means[c])
        nearest_by_dist = int(np.argmin(dist_matrix[c]))
        mapped_arch = arch_map[c]
        flag = "" if nearest_by_dist == mapped_arch else f"  [NOTE: ILR map={mapped_arch}, weight-nearest={nearest_by_dist}]"
        print(f"  Comp {c} -> nearest Arch {nearest_by_dist} (d={dist_matrix[c, nearest_by_dist]:.3f}, center_d={dist_to_center[c]:.3f}){flag}")
        print(f"    weights: [{w_str}]")
        print(f"    distances: [{', '.join(f'{d:.3f}' for d in dist_matrix[c])}]")

    # Heatmap: component x archetype distance
    fig = go.Figure(data=go.Heatmap(
        z=dist_matrix,
        x=[f"Arch {k}" for k in range(K)],
        y=[f"Comp {c} (→A{arch_map[c]})" for c in range(n_comp)],
        colorscale="Viridis_r",
        text=np.round(dist_matrix, 3),
        texttemplate="%{text}",
        colorbar_title="Distance",
    ))
    fig.update_layout(
        title=f"{name}: GMM Component → Archetype Distance",
        xaxis_title="Archetype vertex",
        yaxis_title="GMM Component",
        height=300 + 30 * n_comp,
        width=500,
    )
    fig.show()

    # Component weight profile (stacked bar)
    fig2 = go.Figure()
    for k in range(K):
        fig2.add_trace(go.Bar(
            name=f"Arch {k}",
            x=[f"Comp {c}" for c in range(n_comp)],
            y=weight_means[:, k],
        ))
    fig2.update_layout(
        barmode="stack",
        title=f"{name}: GMM Component Weight Profiles",
        xaxis_title="Component",
        yaxis_title="Mean archetype weight",
        yaxis_range=[0, 1.05],
        height=400,
        width=600,
    )
    fig2.show()

    assert dist_matrix.shape == (n_comp, K)
    assert np.all(dist_matrix >= 0)

## 6. Archetype Comparison**Within-fit MMD:** Test that archetypes are statistically distinct within each cell type.**Between-fit MMD:** Compare archetype distributions across cell types.**Feature similarity:** Silhouette scores for archetype-specific feature profiles.**Wald contrasts:** Pairwise differential gene expression between archetypes.

In [ ]:
# 6a. Within-fit MMD
for name, ad_obj in [("CMP", adata_cmp), ("Mono", adata_mono)]:
    mmd_result = pc.tl.archetype_mmd(ad_obj, n_permutations=100)
    mask = ~np.eye(K, dtype=bool)
    mmd_vals = mmd_result["mmd_matrix"][mask]
    pvals = mmd_result["pvalue_matrix"][mask]

    print(f"\n--- Within-fit MMD ({name}) ---")
    print(f"  MMD range: {mmd_vals.min():.4f} - {mmd_vals.max():.4f}")
    print(f"  Significant pairs: {int(np.sum(pvals < ALPHA))}/{len(pvals)}")

    assert mmd_result["mmd_matrix"].shape == (K, K)
    assert np.all(mmd_vals >= 0)

    _ = pc.pl.mmd_heatmap(ad_obj, show=True)

In [ ]:
# 6b. Between-fit MMD
mmd_between = pc.tl.archetype_mmd(adata_cmp, adata_mono, n_permutations=100)
print("Between-fit MMD (CMP vs Mono):")
print(np.array2string(np.asarray(mmd_between["mmd_matrix"]), precision=4))

In [ ]:
# 6c. Feature similarity
for name, ad_obj in [("CMP", adata_cmp), ("Mono", adata_mono)]:
    sim = pc.tl.archetype_feature_similarity(ad_obj)
    print(f"\n--- Feature Similarity ({name}) ---")
    print(f"  Overall silhouette: {sim['silhouette_overall']:.3f}")
    print(f"  Per-archetype: {np.array2string(np.asarray(sim['silhouette_per_archetype']), precision=3)}")
    _ = pc.pl.feature_similarity_heatmap(ad_obj, show=True)

# Between-fit
sim_between = pc.tl.archetype_feature_similarity(adata_cmp, adata_mono)
print(f"\nBetween-fit Spearman matrix:")
print(np.array2string(np.asarray(sim_between["spearman_matrix"]), precision=3))

In [ ]:
# 6e. Wald contrasts
for name, ad_obj in [("CMP", adata_cmp), ("Mono", adata_mono)]:
    contrasts = pc.tl.archetype_contrasts(ad_obj)
    print(f"\n--- Wald Contrasts ({name}) ---")
    for pair in contrasts["pairs"]:
        key = str(tuple(pair)) if not isinstance(pair, str) else pair
        pvals_fdr = contrasts["pvalues_fdr"][key]
        n_sig = int(np.sum(pvals_fdr < ALPHA))
        print(f"  Pair {pair}: {n_sig}/{len(pvals_fdr)} significant (FDR<{ALPHA})")

    _ = pc.pl.contrast_volcano(ad_obj, pair=(0, 1), show=True)

## 7. HALLMARK Pathway Scoring & RegressionScore cells on MSigDB HALLMARK gene sets using decoupler, then regress pathway scores on archetype weights.This tells us which biological programs are associated with which archetypes.

In [ ]:
net = pc.pp.load_pathway_networks(["hallmark"], verbose=False)
n_pathways = net["source"].nunique()
print(f"HALLMARK: {n_pathways} pathways, {net['target'].nunique()} genes")

pathway_regs = {}
for name, ad_obj in [("CMP", adata_cmp), ("Mono", adata_mono)]:
    pc.pp.compute_pathway_scores(ad_obj, net, verbose=False)
    pathway_names = ad_obj.uns["pathway_scores_pathways"]
    scores = ad_obj.obsm["pathway_scores"]
    print(f"\n--- Pathway Scoring ({name}): {scores.shape[1]} pathways ---")
    print(f"  Score range: [{scores.min():.4f}, {scores.max():.4f}]")
    print(f"  Most variable: {pathway_names[scores.var(axis=0).argmax()]}")

    assert scores.shape[0] == ad_obj.n_obs
    assert np.all(np.isfinite(scores))

    # Pathway regression
    t1 = time.time()
    pw_reg = pc.tl.pathway_simplex_regression(
        ad_obj, n_bootstrap=N_BOOTSTRAP, robust_se=True,
        feature_names=list(pathway_names),
    )
    elapsed = time.time() - t1
    pathway_regs[name] = pw_reg

    pw_r2 = np.asarray(pw_reg["r_squared_degree1"])
    pw_pval_fdr = np.asarray(pw_reg["f_pvalue_fdr"])
    n_sig_pw = int(np.sum(pw_pval_fdr < ALPHA))
    print(f"  Regression ({elapsed:.1f}s): {n_sig_pw}/{len(pw_r2)} significant")

    print(f"  TOP 10 PATHWAYS:")
    for i in np.argsort(pw_r2)[-10:][::-1]:
        print(f"    {pw_reg['feature_names'][i]}: R2={pw_r2[i]:.4f}, p={pw_reg['f_pvalue'][i]:.2e}")

    assert pw_reg["feature_names"][0] != "feature_0", "Generic feature names returned"

In [ ]:
# --- Pathway data structure inspection ---
for name, ad_obj in [("CMP", adata_cmp), ("Mono", adata_mono)]:
    print(f"\n=== {name} Pathway Data ===")
    print(f"  obsm keys: {list(ad_obj.obsm.keys())}")
    scores = ad_obj.obsm["pathway_scores"]
    print(f"  pathway_scores: {scores.shape} dtype={scores.dtype}")
    print(f"  uns keys (simplex/pathway):", [k for k in sorted(ad_obj.uns.keys()) if "simplex" in k or "pathway" in k])

    pw_reg = pathway_regs[name]
    print(f"\n  PathwayRegressionResult keys: {sorted(pw_reg.keys())}")
    print(f"  feature_names[:5]: {pw_reg['feature_names'][:5]}")
    pw_r2 = np.asarray(pw_reg["r_squared_degree1"])
    print(f"  R2 range: [{pw_r2.min():.4f}, {pw_r2.max():.4f}]")

    # Verify gene regression survived pathway regression
    gene_key = "peach_simplex_regression_genes"
    pw_key = "peach_simplex_regression_pathways"
    print(f"\n  {gene_key}: {'PRESENT' if gene_key in ad_obj.uns else 'MISSING'}")
    print(f"  {pw_key}: {'PRESENT' if pw_key in ad_obj.uns else 'MISSING'}")

## 8. Flow Matching (CMP -> Mono)Train a continuous normalizing flow to transport CMP cells to Monocyte distribution in PCA space.Then align genes and HALLMARK pathways with the learned flow direction.**Controls:**- MMD reduction >20% (flow converged meaningfully)- Non-trivial alignment scores (flow velocity is not degenerate)- Gene symbol overlap >10% with HALLMARK- Permutation null for geneset alignment significance

In [ ]:
# Build combined adata for flow
adata_cmp_flow = ad.AnnData(
    X=sp.csr_matrix((adata_cmp.n_obs, 1)),
    obs=adata_cmp.obs.copy(),
)
adata_cmp_flow.obsm["X_pca"] = cell_data["CMP"]["pca"]
adata_cmp_flow.obs["cell_type_label"] = "CMP"

adata_mono_flow = ad.AnnData(
    X=sp.csr_matrix((adata_mono.n_obs, 1)),
    obs=adata_mono.obs.copy(),
)
adata_mono_flow.obsm["X_pca"] = cell_data["Mono"]["pca"]
adata_mono_flow.obs["cell_type_label"] = "Mono"

adata_combined = ad.concat([adata_cmp_flow, adata_mono_flow])
print(f"Combined: {adata_combined.n_obs:,} cells")

# Train flow
print(f"Training flow CMP -> Mono ({FLOW_EPOCHS} epochs)...")
t1 = time.time()
flow_result = pc.tl.flow_within(
    adata_combined,
    source={"cell_type_label": "CMP"},
    target={"cell_type_label": "Mono"},
    pca_key="X_pca",
    hidden_dims=(64, 64, 64),
    n_epochs=FLOW_EPOCHS,
    batch_size=256,
    n_steps=50,
    device="cpu",
)
elapsed = time.time() - t1
mmd_reduction = (flow_result["mmd_before"] - flow_result["mmd_after"]) / flow_result["mmd_before"] * 100

print(f"\nFlow training ({elapsed:.1f}s):")
print(f"  MMD: {flow_result['mmd_before']:.4f} -> {flow_result['mmd_after']:.4f} ({mmd_reduction:.1f}% reduction)")

assert flow_result["mmd_after"] < flow_result["mmd_before"]
assert mmd_reduction > 20.0, f"MMD reduction too small: {mmd_reduction:.1f}%"
assert flow_result["transported"].shape[1] == N_PCS

In [ ]:
# --- Flow result structure inspection ---
print("=== Flow Result ===")
print(f"  type: {type(flow_result).__name__}")
print(f"  keys: {sorted(flow_result.keys())}")
for k, v in sorted(flow_result.items()):
    if isinstance(v, np.ndarray):
        print(f"    {k}: ndarray {v.shape} dtype={v.dtype}")
    elif isinstance(v, (int, float)):
        print(f"    {k}: {v:.6f}")
    elif isinstance(v, list):
        print(f"    {k}: list len={len(v)}")
    else:
        print(f"    {k}: {type(v).__name__}")

print(f"\n  adata_combined:")
print(f"    shape: {adata_combined.shape}")
print(f"    obsm:  {list(adata_combined.obsm.keys())}")
print(f"    obs:   {list(adata_combined.obs.columns)}")

In [ ]:
# Flow visualizations
_ = pc.pl.flow_magnitude(adata_combined, flow_result, show=True)
_ = pc.pl.density_comparison(adata_combined, flow_result, show=True)
_ = pc.pl.velocity_quiver(adata_combined, flow_result, show=True)

In [ ]:
# Gene-level flow alignment
n_top_genes = 50
adata_for_alignment = ad.AnnData(
    X=sp.csr_matrix((adata_combined.n_obs, len(var_names))),
    var=pd.DataFrame(index=var_names),
)
adata_for_alignment.obsm["X_pca"] = adata_combined.obsm["X_pca"]
if pca_loadings is not None:
    adata_for_alignment.varm["PCs"] = pca_loadings
alignment = pc.tl.flow_gene_alignment(adata_for_alignment, flow_result, n_top=n_top_genes)
scores_align = alignment["alignment_scores"]

print(f"Genes scored: {len(scores_align)}")
print(f"Score range: [{scores_align.min():.4f}, {scores_align.max():.4f}]")

print(f"\nTOP 10 FLOW-ALIGNED (upregulated CMP->Mono):")
for gene in alignment["top_aligned"][:10]:
    idx = list(alignment["gene_names"]).index(gene)
    print(f"  {gene}: {scores_align[idx]:.4f}")

print(f"\nTOP 10 FLOW-OPPOSED (downregulated CMP->Mono):")
for gene in alignment["top_opposed"][:10]:
    idx = list(alignment["gene_names"]).index(gene)
    print(f"  {gene}: {scores_align[idx]:.4f}")

assert len(alignment["top_aligned"]) == n_top_genes
assert len(scores_align) == len(var_names)
assert np.abs(scores_align).max() > 1e-6, "Alignment scores all zero"

In [ ]:
# Geneset-level flow alignment (HALLMARK) with permutation null
gene_to_score = dict(zip(alignment["gene_names"], scores_align))
pathway_gene_sets = net.groupby("source")["target"].apply(set).to_dict()

total_pw_genes = sum(len(gs) for gs in pathway_gene_sets.values())
matched_pw_genes = sum(1 for gs in pathway_gene_sets.values() for g in gs if g in gene_to_score)
overlap_rate = matched_pw_genes / total_pw_genes
print(f"Gene overlap: {matched_pw_genes}/{total_pw_genes} ({100*overlap_rate:.1f}%)")
assert overlap_rate > 0.1, f"Overlap too low: {100*overlap_rate:.1f}%"

pathway_alignment = {}
for pw_name, pw_genes in pathway_gene_sets.items():
    matched = [gene_to_score[g] for g in pw_genes if g in gene_to_score]
    if len(matched) >= 5:
        pathway_alignment[pw_name] = {
            "mean_score": float(np.mean(matched)),
            "n_genes": len(matched),
            "n_aligned": sum(1 for s in matched if s > 0),
            "n_opposed": sum(1 for s in matched if s < 0),
        }

# Permutation null
all_scores_list = list(gene_to_score.values())
rng_gs = np.random.default_rng(SEED)
n_perm_gs = 500
for pw_name in pathway_alignment:
    n_g = pathway_alignment[pw_name]["n_genes"]
    null_means = np.array([
        float(np.mean(rng_gs.choice(all_scores_list, size=n_g, replace=False)))
        for _ in range(n_perm_gs)
    ])
    obs_mean = pathway_alignment[pw_name]["mean_score"]
    pval_gs = (np.sum(np.abs(null_means) >= np.abs(obs_mean)) + 1) / (n_perm_gs + 1)
    pathway_alignment[pw_name]["pvalue"] = float(pval_gs)

n_sig_gs = sum(1 for v in pathway_alignment.values() if v["pvalue"] < ALPHA)
print(f"\nPathways scored: {len(pathway_alignment)}, significant: {n_sig_gs}")

print(f"\nTOP 10 FLOW-ALIGNED PATHWAYS (activated CMP->Mono):")
for pw, info in sorted(pathway_alignment.items(), key=lambda x: x[1]["mean_score"], reverse=True)[:10]:
    print(f"  {pw}: mean={info['mean_score']:.4f} ({info['n_aligned']}/{info['n_genes']} aligned, p={info['pvalue']:.3f})")

print(f"\nTOP 10 FLOW-OPPOSED PATHWAYS (deactivated CMP->Mono):")
for pw, info in sorted(pathway_alignment.items(), key=lambda x: x[1]["mean_score"])[:10]:
    print(f"  {pw}: mean={info['mean_score']:.4f} ({info['n_opposed']}/{info['n_genes']} opposed, p={info['pvalue']:.3f})")

assert len(pathway_alignment) > 0

## 9. Summary & Cross-ValidationCross-cell-type concordance checks and final pipeline summary.

In [ ]:
from scipy.stats import spearmanr

# Gene R2 concordance
r2_cmp = np.asarray(gene_regs["CMP"]["r_squared_degree1"])
r2_mono = np.asarray(gene_regs["Mono"]["r_squared_degree1"])
rho, pval = spearmanr(r2_cmp, r2_mono)
print(f"Gene R2 concordance: rho={rho:.4f} (p={pval:.2e})")
assert rho > 0 and pval < 0.05

# Pathway R2 concordance
pw_r2_cmp = np.asarray(pathway_regs["CMP"]["r_squared_degree1"])
pw_r2_mono = np.asarray(pathway_regs["Mono"]["r_squared_degree1"])
pw_rho, pw_pval = spearmanr(pw_r2_cmp, pw_r2_mono)
print(f"Pathway R2 concordance: rho={pw_rho:.4f} (p={pw_pval:.2e})")

# Summary
elapsed_total = time.time() - t0
print(f"\n{'='*60}")
print(f"  PIPELINE SUMMARY")
print(f"{'='*60}")
print(f"  Cell types: CMP ({adata_cmp.n_obs:,}), Mono ({adata_mono.n_obs:,})")
print(f"  Archetypes: K={K}")
print(f"  CMP R2: {models['CMP'].get('final_archetype_r2', 0):.3f}")
print(f"  Mono R2: {models['Mono'].get('final_archetype_r2', 0):.3f}")
print(f"  Genes tested: {adata_cmp.n_vars:,}")
print(f"  CMP sig (FDR<{ALPHA}): {int(np.sum(np.asarray(gene_regs['CMP']['f_pvalue_fdr']) < ALPHA))}")
print(f"  Mono sig (FDR<{ALPHA}): {int(np.sum(np.asarray(gene_regs['Mono']['f_pvalue_fdr']) < ALPHA))}")
print(f"  HALLMARK pathways: {len(pathway_alignment)} scored, {n_sig_gs} sig")
print(f"  Flow MMD: {flow_result['mmd_before']:.4f} -> {flow_result['mmd_after']:.4f}")
print(f"  Top aligned genes: {alignment['top_aligned'][:5]}")
print(f"  Total time: {elapsed_total:.0f}s ({elapsed_total/60:.1f} min)")
print(f"{'='*60}")
print(f"  ALL SECTIONS PASSED")